# [SK 03.5 - RESPONSES Agent using `GetResponse` (without plugins)](https://learn.microsoft.com/en-us/python/api/semantic-kernel/semantic_kernel.agents.open_ai.azure_responses_agent.azureresponsesagent?view=semantic-kernel-python)

## **Semantic Kernel** [`AzureResponsesAgent`](https://learn.microsoft.com/en-us/python/api/semantic-kernel/semantic_kernel.agents.open_ai.azure_responses_agent.azureresponsesagent?view=semantic-kernel-python) class implements an **Open AI** [`Agent`](https://openai.github.io/openai-agents-python/)
Documentation [here](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/responses-agent?pivots=programming-language-python).

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv # requires python-dotenv

load_dotenv("./../config/credentials_my.env")

agent_name                  = "my_response_agent"
instructions                = "you are a clever agent"

print(f"os.environ['AZURE_OPENAI_ENDPOINT']: {os.environ['AZURE_OPENAI_ENDPOINT']}")

os.environ['AZURE_OPENAI_ENDPOINT']: https://mmoaiswc-01.openai.azure.com/


# Set the logging level for  semantic_kernel.kernel to DEBUG
One of the main benefits of using Semantic Kernel is that it supports enterprise-grade services.<br/>
In this sample, we add the logging service to the kernel to help debug the AI agent.

In [2]:
import logging

logging.basicConfig(
    format="[%(asctime)s - %(name)s:%(lineno)d - %(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S")
    
logging.getLogger("kernel").setLevel(logging.DEBUG)
logging.getLogger().addHandler(logging.StreamHandler())
logging.getLogger().setLevel(logging.ERROR) # or logging.DEBUG to see more details

# Creating an [AzureResponsesAgent](https://learn.microsoft.com/en-us/python/api/semantic-kernel/semantic_kernel.agents.open_ai.azure_responses_agent.azureresponsesagent?view=semantic-kernel-python)

## Creating an AzureResponsesAgent requires first creating a client to be able to talk a remote Azure OpenAI service

In [3]:
from semantic_kernel.agents import AzureResponsesAgent
from semantic_kernel.connectors.ai.open_ai import AzureOpenAISettings

# OpenAI Endpoint, Key and Deployment name can be implicit if the following environment variables are set
os.environ['AZURE_OPENAI_ENDPOINT'] = os.environ['AZURE_OPENAI_ENDPOINT'] # variable already exists, same name
os.environ['AZURE_OPENAI_API_KEY']  = os.environ['AZURE_OPENAI_API_KEY'] # variable already exists, same name
os.environ['AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME'] = os.environ['AZURE_OPENAI_CHAT_DEPLOYMENT_NAME'] # variable already exists, different name

# Set up the client and model using Azure OpenAI Resources
client = AzureResponsesAgent.create_client()

AzureOpenAISettings()

AzureOpenAISettings(env_file_path=None, env_file_encoding='utf-8', chat_deployment_name='gpt-4o', responses_deployment_name='gpt-4o', text_deployment_name='gpt-35-turbo-instruct', embedding_deployment_name='text-embedding-ada-002', text_to_image_deployment_name=None, audio_to_text_deployment_name=None, text_to_audio_deployment_name=None, realtime_deployment_name=None, endpoint=AnyUrl('https://mmoaiswc-01.openai.azure.com/'), base_url=None, api_key=SecretStr('**********'), api_version='2025-04-01-preview', token_endpoint='https://cognitiveservices.azure.com/.default')

## Create the AzureResponsesAgent instance using the client and the deployment name

In [4]:
agent = AzureResponsesAgent(
    ai_model_id=AzureOpenAISettings().responses_deployment_name,
    client=client,
    instructions=instructions,
    name=agent_name
)

agent

AzureResponsesAgent(arguments=None, description=None, id='5d76a833-685f-4ee6-8df3-4c0fa3928f95', instructions='you are a clever agent', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x779f11369be0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[]), name='my_response_agent', prompt_template=None, ai_model_id='gpt-4o', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x779f12d2e660>, function_choice_behavior=FunctionChoiceBehavior(enable_kernel_functions=True, maximum_auto_invoke_attempts=5, filters=None, type_=<FunctionChoiceType.AUTO: 'auto'>), instruction_role='developer', metadata={}, temperature=None, top_p=None, plugins=[], polling_options=RunPollingOptions(default_polling_interval=datetime.timedelta(microseconds=250000), default_polling_backoff=datetime.timedelta(seconds=1), default_polling_backoff_thr

# Using an OpenAIResponsesAgent
The OpenAI Responses API supports optional remote storage of conversations. By default, when using a ResponsesAgentThread, responses are stored remotely. This enables the use of the Responses API's previous_response_id for maintaining context across invocations.<br/>

Each conversation is treated as a thread, identified by a unique string ID. All interactions with your OpenAIResponsesAgent are scoped to this thread identifier.<br/>

The underlying mechanics of the Responses API thread are abstracted by the ResponsesAgentThread class, which implements the AgentThread interface.<br/>

The OpenAIResponsesAgent currently only supports threads of type ResponsesAgentThread.<br/>

You can invoke the OpenAIResponsesAgent without specifying an AgentThread, to start a new thread and a new AgentThread will be returned as part of the response.

## Using the agents with `get_response()`

In [5]:
USER_INPUTS = [
    "My name is John Doe.",
    "Tell me a joke",
    "Explain why this is funny.",
    "What have we been talking about?",
]

thread = None

# Generate the agent response(s)
for user_input in USER_INPUTS:
    print(f"# User: '{user_input}'")
    # Invoke the agent for the current message and print the response
    response = await agent.get_response(messages=user_input, thread=thread)
    print(f"# {response.name}: {response.content}\n")
    # Update the thread so the previous response id is used
    thread = response.thread

# Delete the thread when it is no longer needed
if thread:
    await thread.delete()
    print(f"\nthread <{thread.id}> has been deleted.")

# User: 'My name is John Doe.'
# my_response_agent: Nice to meet you, John! How can I assist you today?

# User: 'Tell me a joke'
# my_response_agent: Sure! Why don't scientists trust atoms?

Because they make up everything!

# User: 'Explain why this is funny.'
# my_response_agent: The joke is funny because it plays on the double meaning of the phrase "make up everything." 

1. In science, atoms are the basic building blocks of matter, meaning they literally compose everything around us.

2. In everyday language, "make up everything" can mean to fabricate stories or tell lies.

The humor comes from the clever wordplay and the unexpected twist that turns a scientific truth into a playful accusation.

# User: 'What have we been talking about?'
# my_response_agent: We've been talking about a joke involving atoms and why it's funny due to its clever wordplay.


thread <resp_68c66edb9cd481979d35c899a88ac20f07df851784e44247> has been deleted.
